<div >
<img src = "figs/ans_banner_1920x200.png" />
</div>

# Caso-taller: Identificando  Burger Master con MMG

El Burger Master es un evento creado en el 2016 por el *influencer* Tulio Zuluaga, más conocido en redes como Tulio recomienda, el cual busca que por una semana las hamburgueserías de cada ciudad ofrezcan su mejor producto a un precio reducido. 

El evento ha venido creciendo y en el 2022 se extendió por 21 ciudades de Colombia para las cuales se estimó que se vendieron más de dos millones de hamburguesas. El objetivo del presente caso-taller  es identificar los puntos calientes de hamburgueserías  que compitieron en  la ciudad de Bogotá aplicando el Modelo de Mezclas Gaussianas.

## Instrucciones generales

1. Para desarrollar el *cuaderno* primero debe descargarlo.

2. Para responder cada inciso deberá utilizar el espacio debidamente especificado.

3. La actividad será calificada sólo si sube el *cuaderno* de jupyter notebook con extensión `.ipynb` en la actividad designada como "Revisión por el compañero."

4. El archivo entregado debe poder ser ejecutado localmente por los pares. Sea cuidadoso con la especificación de la ubicación de los archivos de soporte, guarde la carpeta de datos  en la misma ruta de acceso del cuaderno, por ejemplo: `data`.


## Desarrollo

#### Config

In [ ]:
from __future__ import annotations

%load_ext autoreload
%autoreload 2

# python
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.cm as cm # Para las paletas de colores
from enum import Enum
from matplotlib.colors import to_rgba
import session_info

# tools
from pathlib import Path
from inspect import cleandoc
from dataclasses import dataclass

# stats
import statsmodels.api as sm
from scipy import stats
from statsmodels.nonparametric.kernel_density import KDEMultivariate


# Geo
import folium
import geojsoncontour
import geopandas as gpd
from pyrosm import OSM, get_data
from geopy.distance import geodesic


# sklearn
from sklearn import model_selection
from sklearn.neighbors import KernelDensity
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# view
from IPython.display import IFrame


# typings
from typing import List, Tuple, Dict, Any, Union, Optional

# setup
plt.style.use('seaborn')
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('max_colwidth', None)

# decimals
np.set_printoptions(precision=6)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


##### Información de Sesión

In [7]:
session_info.show(html=False)

-----
folium              0.14.0
geojsoncontour      NA
geopandas           0.10.2
geopy               2.4.0
matplotlib          3.3.1
numpy               1.19.1
pandas              1.1.1
pyrosm              NA
scipy               1.7.3
seaborn             0.12.2
session_info        1.0.0
shapely             1.8.5
sklearn             0.23.2
statsmodels         0.13.5
-----
IPython             7.34.0
jupyter_client      7.4.9
jupyter_core        4.12.0
jupyterlab          2.1.1
notebook            6.0.3
-----
Python 3.7.17 (default, Jun 13 2023, 16:36:15) [GCC 8.3.0]
Linux-5.15.49-linuxkit-x86_64-with-debian-10.13
-----
Session information updated at 2025-09-27 00:02


### utils

In [20]:
# utils
# rutas absolutas: agnostico al sistema operativo
here: Path = Path.cwd().absolute()
data: Path = here / 'data'

burger_master_xlsx: Path = data / 'burger_master.xlsx'

### 1. Carga de datos  

En la carpeta `data` se encuentra el archivo `burger_master.xlsx` para la ciudad de Bogotá, cargue estos datos en su *cuaderno* y reporte brevemente el contenido de la base.

In [86]:
# Utilice este espacio para escribir el código.
bogota_mb_df = pd.read_excel(burger_master_xlsx, engine='openpyxl')
bogota_mb_df.head(3)

,Restaurante,Dirección,Descripción,Latitud,Longitud
0,MAIKKI,Cra 75 # 24D – 48,"<p>MAIKKI MACUIRA: Cama de chicharrón soplado, tocineta caramelizada en melao de maracuyá y panela de la hoya del rio Suárez, carne de res del Magdalena medio, queso fresco de pasta hilada, mayonesa de cilantro cimarrón del pacifico, cogollo de lechuga y pan artesanal. Un homenaje a lo nuestro. En pie de lucha por la comida rápida colombiana<br/><br/>Cra 75 # 24D – 48, Modelia<br/>Calle 119 # 11A – 24, Santa Bárbara<br/>Cra 47A # 98 – 47, Castellana</p>",4.668833,-74.116828
1,MAIKKI,Calle 119 # 11A – 24,"<p>MAIKKI MACUIRA: Cama de chicharrón soplado, tocineta caramelizada en melao de maracuyá y panela de la hoya del rio Suárez, carne de res del Magdalena medio, queso fresco de pasta hilada, mayonesa de cilantro cimarrón del pacifico, cogollo de lechuga y pan artesanal. Un homenaje a lo nuestro. En pie de lucha por la comida rápida colombiana<br/><br/>Cra 75 # 24D – 48, Modelia<br/>Calle 119 # 11A – 24, Santa Bárbara<br/>Cra 47A # 98 – 47, Castellana</p>",4.698395,-74.036585
2,MAIKKI,Cra 47A # 98 – 47,"<p>MAIKKI MACUIRA: Cama de chicharrón soplado, tocineta caramelizada en melao de maracuyá y panela de la hoya del rio Suárez, carne de res del Magdalena medio, queso fresco de pasta hilada, mayonesa de cilantro cimarrón del pacifico, cogollo de lechuga y pan artesanal. Un homenaje a lo nuestro. En pie de lucha por la comida rápida colombiana<br/><br/>Cra 75 # 24D – 48, Modelia<br/>Calle 119 # 11A – 24, Santa Bárbara<br/>Cra 47A # 98 – 47, Castellana</p>",4.686401,-74.060144


In [87]:
print(bogota_mb_df.shape)
bogota_mb_df.info()

(137, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 137 entries, 0 to 136
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Restaurante  137 non-null    object 
 1   Dirección    137 non-null    object 
 2   Descripción  137 non-null    object 
 3   Latitud      137 non-null    float64
 4   Longitud     137 non-null    float64
dtypes: float64(2), object(3)
memory usage: 5.5+ KB


(Utilice este espacio para describir el procedimiento, análisis, y conclusiones)

### 2.  Visualizando los datos

Visualice la ubicación de cada restaurante en un mapa interactivo. Añada un marcador para cada restaurante y la posibilidad de encontrar la descripción de la hamburguesa ofrecida en un pop-up. (Note que la columna Descripción contiene otra información adicional).

In [88]:
# Utilice este espacio para escribir el código.
bogota_mb_lat = bogota_mb_df.Latitud
bogota_mb_lon = bogota_mb_df.Longitud
bogota_mb_location = [bogota_mb_lat.mean(), bogota_mb_lon.mean()]

mapa_bogota_mb = folium.Map(
    location=bogota_mb_location,
    zoom_start=10,
    tiles="cartodbpositron"
)

lat_lon_des = zip(
    bogota_mb_lat,
    bogota_mb_lon, 
    bogota_mb_df['Descripción']
)

for lat, lon, desc in lat_lon_des:
   folium.Marker(
      location=[lat, lon],
      tooltip=desc,
   ).add_to(mapa_bogota_mb)
   
mapa_bogota_mb


(Utilice este espacio para describir el procedimiento, análisis, y conclusiones)

### 3.  Análisis de puntos calientes

Aplique el modelo de Mezclas Gaussianas para buscar clusters de restaurantes en Bogotá, mencione qué estructura de covarianza usó y explique por qué. Escoja el número óptimo de componentes, explicando el procedimiento y justificando su elección.

In [80]:
# Utilice este espacio para escribir el código.
# Bajamos los datos para  Bogotá
# fp = get_data("Bogota")
# Inicializamos el lector para Bogotá
# osm = OSM(fp)
# restaurants = osm.get_pois(custom_filter={"amenity": ["restaurant"]})
# capital = osm.get_boundaries(boundary_type='administrative', name="Bogotá")
# capital.plot()


In [89]:
bogota_mb_df = gpd.GeoDataFrame(
    bogota_mb_df, geometry = gpd.points_from_xy(bogota_mb_lon, bogota_mb_lat)
)
bogota_mb_df.crs = "EPSG:4326"
# restaurants = gpd.sjoin(db, capital)

In [90]:
X = bogota_mb_df[['Longitud', 'Latitud']].values
MMG_mb = GaussianMixture(n_components=5, covariance_type='full',random_state=123)
labels = MMG_mb .fit(X).predict(X)

In [91]:
bogota_mb_df.loc[:,'cluster'] = labels

In [93]:
map = folium.Map(
    location = bogota_mb_location, 
    tiles="cartodb positron",
    # crs="EPSG:4326",
    zoom_start = 10
)

class ClusterColor(Enum):
    CLUSTER_0 = 'red'
    CLUSTER_1 = 'orange'
    CLUSTER_2 = 'green'
    CLUSTER_3 = 'purple'
    CLUSTER_4 = 'cadetblue'

lat_lon_lab = zip(
    bogota_mb_lat,
    bogota_mb_lon,
    labels
)

#capa clusters de teatros
for lat, lon, c in lat_lon_lab:
    color:str = ClusterColor[f'CLUSTER_{c}'].value
    folium.CircleMarker(
        [lat, lon],
        radius=8,
        fill_color=color,
        fill=True,
        color=color,
        fill_opacity=0.3,
        weight=0
        ).add_to(map)
    
map

(Utilice este espacio para describir el procedimiento, análisis, y conclusiones)

#### 3.1. Visualización de los resultados

Visualice las densidades estimadas por el  mejor modelo estimado en la sección anterior usando un mapa de calor interactivo, discuta los resultados.

In [ ]:
# Utilice este espacio para escribir el código.

(Utilice este espacio para describir el procedimiento, análisis, y conclusiones)

### 4. Comparación con KDE

Estime ahora las densidades usando KDE bivariado de la librería `statsmodels` con el anchos de banda dado por `cv_ml`. Muestre los resultados usando un mapa interactivo. Compare los resultados obtenidos por el "mejor" modelo encontrado via MMG. 

In [ ]:
# Utilice este espacio para escribir el código.

(Utilice este espacio para describir el procedimiento, análisis, y conclusiones)